# APISR DAT 4× 推理 (Inference)

本 Notebook 集成 [APISR](https://github.com/Kiteretsu77/APISR) 的 **DAT**（Dual Aggregation Transformer）预训练模型，对低分辨率图像做 **4× 超分**。DAT 为论文/Model Zoo 提供的 4× 架构之一。

- **权重**：从 APISR Model Zoo 下载 `4x_APISR_DAT_GAN_generator.pth` 放入 `pretrained_models/`
- **输入**：`dataset/lowres_4x/original`
- **输出**：`results/APISR_DAT_4x_inference`

In [ ]:
import os
import sys
import glob
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision.transforms import ToTensor

apisr_tools_path = os.path.abspath('APISR_tools')
if apisr_tools_path not in sys.path:
    sys.path.insert(0, apisr_tools_path)
from architecture.dat import DAT

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

In [ ]:
SCALE = 4
MODEL_PATH = 'pretrained_models/4x_APISR_DAT_GAN_generator.pth'
LR_DIR = 'dataset/lowres_4x/original'
HR_DIR = 'dataset/highres/original'
OUTPUT_DIR = 'results/APISR_DAT_4x_inference'
TILE = 256
TILE_PAD = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Model:', MODEL_PATH)

In [ ]:
# DAT small (与 APISR test_utils.load_dat 一致)
generator = DAT(
    upscale=SCALE,
    in_chans=3,
    img_size=64,
    img_range=1.,
    depth=[6, 6, 6, 6, 6, 6],
    embed_dim=180,
    num_heads=[6, 6, 6, 6, 6, 6],
    expansion_factor=2,
    resi_connection='1conv',
    split_size=[8, 16],
    upsampler='pixelshuffledirect',
).to(device)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f'未找到权重: {MODEL_PATH}\n请从 APISR Model Zoo 下载 4x_APISR_DAT_GAN_generator.pth')

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
if 'model_state_dict' not in ckpt:
    raise KeyError('Checkpoint 需包含 model_state_dict')
weight = ckpt['model_state_dict'].copy()
for old_key in list(weight.keys()):
    if old_key.startswith('_orig_mod.'):
        weight[old_key[10:]] = weight.pop(old_key)
generator.load_state_dict(weight, strict=True)
generator.eval()
print('DAT 模型加载完成。')

In [ ]:
def infer_one(lr_path, save_dir=None, tile=0, tile_pad=16):
    img = cv2.imread(lr_path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if h % 4 != 0:
        img = img[:4*(h//4), :, :]
    if w % 4 != 0:
        img = img[:, :4*(w//4), :]
    h, w = img.shape[:2]
    x = ToTensor()(img).unsqueeze(0).to(device)

    if tile > 0 and (h > tile or w > tile):
        out = torch.zeros(1, 3, h * SCALE, w * SCALE, device='cpu', dtype=torch.float32)
        weight = torch.zeros(1, 1, h * SCALE, w * SCALE, device='cpu', dtype=torch.float32)
        for y in range(0, h, tile):
            for x0 in range(0, w, tile):
                y1, x1 = min(y + tile, h), min(x0 + tile, w)
                y0p = max(y - tile_pad, 0)
                x0p = max(x0 - tile_pad, 0)
                y1p = min(y1 + tile_pad, h)
                x1p = min(x1 + tile_pad, w)
                patch = x[:, :, y0p:y1p, x0p:x1p].to(device)
                with torch.no_grad():
                    pred = generator(patch).float().cpu()
                th, tw = (y1 - y) * SCALE, (x1 - x0) * SCALE
                oy, ox = y * SCALE, x0 * SCALE
                out[:, :, oy:oy+th, ox:ox+tw] += pred[:, :, (y-y0p)*SCALE:(y1-y0p)*SCALE, (x0-x0p)*SCALE:(x1-x0p)*SCALE]
                weight[:, :, oy:oy+th, ox:ox+tw] += 1.0
        sr = (out / weight.clamp(min=1e-6)).clamp(0, 1)
    else:
        with torch.no_grad():
            sr = generator(x).float().cpu().clamp(0, 1)

    sr_np = (sr.squeeze(0).permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    sr_bgr = cv2.cvtColor(sr_np, cv2.COLOR_RGB2BGR)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        base = os.path.splitext(os.path.basename(lr_path))[0]
        cv2.imwrite(os.path.join(save_dir, f'{base}.png'), sr_bgr)
    return sr_bgr

In [ ]:
from tqdm import tqdm
lr_paths = sorted(glob.glob(os.path.join(LR_DIR, '*.*')))
if not lr_paths:
    raise FileNotFoundError(f'未找到 LR 图像: {LR_DIR}')
num_vis = min(3, len(lr_paths))
vis_idx = set(random.sample(range(len(lr_paths)), num_vis))
for i, p in enumerate(tqdm(lr_paths, desc='APISR DAT 4x')):
    out = infer_one(p, save_dir=OUTPUT_DIR, tile=TILE, tile_pad=TILE_PAD)
    if out is not None and i in vis_idx:
        lr = cv2.imread(p)
        lr = cv2.cvtColor(lr, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(lr); ax[0].set_title('LR'); ax[0].axis('off')
        ax[1].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); ax[1].set_title('SR (APISR DAT)'); ax[1].axis('off')
        plt.tight_layout(); plt.show()
print('批量推理完成。')

In [ ]:
# 可选：NIQE / MANIQA / CLIPIQA
try:
    import pyiqa
except ImportError:
    print('未安装 pyiqa，跳过评估。')
else:
    sr_list = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.*')))
    if sr_list:
        dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        niqe_m = pyiqa.create_metric('niqe', device=dev)
        maniqa_m = pyiqa.create_metric('maniqa', device=dev)
        clipiqa_m = pyiqa.create_metric('clipiqa', device=dev)
        n = [niqe_m(p).item() for p in tqdm(sr_list, desc='NIQE')]
        ma = [maniqa_m(p).item() for p in tqdm(sr_list, desc='MANIQA')]
        c = [clipiqa_m(p).item() for p in tqdm(sr_list, desc='CLIPIQA')]
        summary = f'APISR DAT — NIQE(↓): {np.mean(n):.4f}  MANIQA(↑): {np.mean(ma):.4f}  CLIPIQA(↑): {np.mean(c):.4f}'
        print(summary)
        with open(os.path.join(OUTPUT_DIR, 'scores.txt'), 'w') as f:
            f.write(summary + '\n')
        print('(Summary also saved to', os.path.join(OUTPUT_DIR, 'scores.txt') + ')')